In [1]:

import numpy as np
import pandas as pd
from pathlib import Path

from scipy.stats import skew, kurtosis
from scipy.signal import welch
from scipy.integrate import trapezoid

# Konfigurasi
FS = 12000
WINDOW_SIZE = 1200

DATA_PATH = Path("outputs/bearing_filtered.csv")
OUTPUT_PATH = Path("outputs/bearing_features.csv")

print("Feature Engineering siap dijalankan.")


Feature Engineering siap dijalankan.


In [2]:
# Membaca dan memvalidasi dataset
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"File tidak ditemukan: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

required_columns = [
    "source_file",
    "condition",
    "defect_size_inch",
    "motor_load_hp",
    "sample_index",
    "sampling_rate_hz",
    "vibration_filtered"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(f"Kolom tidak tersedia: {missing}")

if df["source_file"].nunique() != 40:
    raise ValueError(
        "Jumlah rekaman tidak sesuai. Diharapkan 40 file."
    )

if not np.allclose(df["sampling_rate_hz"], FS):
    raise ValueError("Sampling rate tidak sesuai.")

print(f"Jumlah sampel: {len(df):,}")
print(f"Jumlah rekaman: {df['source_file'].nunique()}")
print(f"Durasi per segmen: {WINDOW_SIZE / FS:.2f} detik")


Jumlah sampel: 4,815,787
Jumlah rekaman: 40
Durasi per segmen: 0.10 detik


In [3]:
# Fungsi ekstraksi fitur domain waktu
def extract_time_features(signal):
    signal = np.asarray(signal, dtype=float)

    rms = np.sqrt(np.mean(signal ** 2))
    peak = np.max(np.abs(signal))

    return {
        "mean": np.mean(signal),
        "std": np.std(signal),
        "rms": rms,
        "skewness": skew(signal),
        "kurtosis": kurtosis(signal, fisher=True),
        "peak_to_peak": np.ptp(signal),
        "peak_amplitude": peak,
        "crest_factor": peak / rms if rms > 0 else 0.0
    }

print("Fungsi fitur domain waktu berhasil dibuat.")


Fungsi fitur domain waktu berhasil dibuat.


In [4]:
# Fungsi ekstraksi fitur domain frekuensi
def extract_frequency_features(signal, fs=12000):
    signal = np.asarray(signal, dtype=float)

    freq, psd = welch(
        signal,
        fs=fs,
        window="hann",
        nperseg=min(1024, len(signal)),
        noverlap=min(512, len(signal) // 2)
    )

    psd = np.maximum(psd, 0)

    total_power = trapezoid(psd, freq)
    power_sum = np.sum(psd)

    if power_sum > 0:
        probability = psd / power_sum
        centroid = np.sum(freq * probability)
        bandwidth = np.sqrt(
            np.sum(((freq - centroid) ** 2) * probability)
        )
        entropy = -np.sum(
            probability[probability > 0] *
            np.log2(probability[probability > 0])
        )
    else:
        centroid = 0.0
        bandwidth = 0.0
        entropy = 0.0

    dominant_frequency = freq[np.argmax(psd)]

    def band_power(low, high):
        mask = (freq >= low) & (freq <= high)
        return trapezoid(psd[mask], freq[mask])

    return {
        "dominant_frequency": dominant_frequency,
        "spectral_centroid": centroid,
        "spectral_bandwidth": bandwidth,
        "spectral_entropy": entropy,
        "total_spectral_power": total_power,
        "band_power_low": band_power(0, 1000),
        "band_power_mid": band_power(1000, 3000),
        "band_power_high": band_power(3000, 6000)
    }

print("Fungsi fitur domain frekuensi berhasil dibuat.")


Fungsi fitur domain frekuensi berhasil dibuat.


In [5]:
# Segmentasi dan ekstraksi fitur seluruh 40 file
feature_records = []
segmentation_report = []

for source_file, group in df.groupby(
    "source_file", sort=False
):
    group = group.sort_values("sample_index")

    signal = group[
        "vibration_filtered"
    ].to_numpy(dtype=float)

    if not np.isfinite(signal).all():
        raise ValueError(
            f"Sinyal tidak valid pada {source_file}"
        )

    condition = group["condition"].iloc[0]
    defect_size = group["defect_size_inch"].iloc[0]
    motor_load = group["motor_load_hp"].iloc[0]

    # Jumlah segmen lengkap
    n_segments = len(signal) // WINDOW_SIZE

    for segment_id in range(n_segments):

        start = segment_id * WINDOW_SIZE
        end = start + WINDOW_SIZE

        segment = signal[start:end]

        # Ekstraksi fitur
        time_features = extract_time_features(segment)
        frequency_features = extract_frequency_features(
            segment, FS
        )

        record = {
            "source_file": source_file,
            "segment_id": segment_id,
            "start_sample": start,
            "end_sample": end - 1,
            "condition": condition,
            "defect_size_inch": defect_size,
            "motor_load_hp": motor_load,
            **time_features,
            **frequency_features
        }

        feature_records.append(record)

    segmentation_report.append({
        "source_file": source_file,
        "total_samples": len(signal),
        "total_segments": n_segments,
        "unused_samples": len(signal) % WINDOW_SIZE
    })

    print(
        f"{source_file}: "
        f"{n_segments} segmen berhasil diproses"
    )

features_df = pd.DataFrame(feature_records)
segmentation_df = pd.DataFrame(segmentation_report)

print("\nFeature Engineering selesai!")
print(f"Jumlah segmen: {len(features_df):,}")
print(f"Jumlah fitur numerik: 16")


97.mat: 50 segmen berhasil diproses
98 .mat: 100 segmen berhasil diproses
99.mat: 101 segmen berhasil diproses
100.mat: 101 segmen berhasil diproses
105.mat: 101 segmen berhasil diproses
106.mat: 101 segmen berhasil diproses
107.mat: 101 segmen berhasil diproses
108.mat: 102 segmen berhasil diproses
118.mat: 102 segmen berhasil diproses
119.mat: 101 segmen berhasil diproses
120.mat: 101 segmen berhasil diproses
121.mat: 101 segmen berhasil diproses
130.mat: 101 segmen berhasil diproses
131.mat: 102 segmen berhasil diproses
132.mat: 101 segmen berhasil diproses
133.mat: 102 segmen berhasil diproses
169.mat: 101 segmen berhasil diproses
170.mat: 101 segmen berhasil diproses
171.mat: 101 segmen berhasil diproses
172.mat: 101 segmen berhasil diproses
185.mat: 101 segmen berhasil diproses
186.mat: 101 segmen berhasil diproses
187.mat: 101 segmen berhasil diproses
188.mat: 101 segmen berhasil diproses
197.mat: 101 segmen berhasil diproses
198.mat: 101 segmen berhasil diproses
199.mat: 101 se

In [6]:
# Validasi hasil feature engineering
feature_columns = [
    "mean", "std", "rms", "skewness",
    "kurtosis", "peak_to_peak",
    "peak_amplitude", "crest_factor",
    "dominant_frequency", "spectral_centroid",
    "spectral_bandwidth", "spectral_entropy",
    "total_spectral_power",
    "band_power_low", "band_power_mid",
    "band_power_high"
]

invalid_count = (
    ~np.isfinite(
        features_df[feature_columns].to_numpy(dtype=float)
    )
).sum()

print("Jumlah nilai fitur tidak valid:", invalid_count)

if invalid_count > 0:
    raise ValueError(
        "Ditemukan nilai fitur NaN atau Infinity."
    )

print("\nDistribusi segmen berdasarkan kondisi:")
display(features_df["condition"].value_counts())

print("\nDistribusi segmen berdasarkan beban motor:")
display(features_df["motor_load_hp"].value_counts())

print("\nContoh dataset fitur:")
display(features_df.head())


Jumlah nilai fitur tidak valid: 0

Distribusi segmen berdasarkan kondisi:


condition
Outer Race    1215
Inner Race    1213
Ball Fault    1213
Normal         352
Name: count, dtype: int64


Distribusi segmen berdasarkan beban motor:


motor_load_hp
3    1012
1    1010
2    1010
0     961
Name: count, dtype: int64


Contoh dataset fitur:


,source_file,segment_id,start_sample,end_sample,condition,defect_size_inch,motor_load_hp,mean,std,rms,...,peak_amplitude,crest_factor,dominant_frequency,spectral_centroid,spectral_bandwidth,spectral_entropy,total_spectral_power,band_power_low,band_power_mid,band_power_high
0,97.mat,0,0,1199,Normal,NaN,0,-0.000437,0.072389,0.072390,...,0.216706,2.993599,4148.4375,3076.863339,1588.327631,5.084973,0.004946,0.001197,0.000463,0.003284
1,97.mat,1,1200,2399,Normal,NaN,0,0.000018,0.069990,0.069990,...,0.242119,3.459334,4148.4375,3169.305563,1537.997776,5.200270,0.004990,0.001109,0.000463,0.003413
2,97.mat,2,2400,3599,Normal,NaN,0,-0.000180,0.068403,0.068404,...,0.247259,3.614699,4148.4375,3291.095042,1460.725365,5.149379,0.004693,0.000829,0.000444,0.003418
3,97.mat,3,3600,4799,Normal,NaN,0,0.000025,0.070061,0.070061,...,0.244666,3.492195,4148.4375,3225.832963,1485.812198,5.089351,0.004650,0.000933,0.000481,0.003236
4,97.mat,4,4800,5999,Normal,NaN,0,-0.000124,0.069863,0.069863,...,0.210170,3.008313,4148.4375,3150.739684,1549.269331,5.254523,0.004976,0.001069,0.000460,0.003446


In [7]:
# Menyimpan dataset fitur dan laporan segmentasi
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

features_df.to_csv(
    OUTPUT_PATH,
    index=False
)

segmentation_df.to_csv(
    "outputs/segmentation_report.csv",
    index=False
)

print("Dataset fitur berhasil disimpan!")
print(f"Lokasi: {OUTPUT_PATH.resolve()}")
print(f"Jumlah segmen: {len(features_df):,}")
print(f"Jumlah rekaman: {features_df['source_file'].nunique()}")


Dataset fitur berhasil disimpan!
Lokasi: D:\python\sem 5\Pemrosesan sinyal\Tugas Besar\outputs\bearing_features.csv
Jumlah segmen: 3,993
Jumlah rekaman: 40
